In [ ]:
# !pip install tensorflow
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# prompt: how to download this notebook

from google.colab import files
files.download('your_notebook_name.ipynb')
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
dataset_train_file_path_leaf = r"/storage/dataset/train"
dataset_val_file_path_leaf = r"/storage/dataset/val"
dataset_test_file_path_leaf = r"/storage/dataset/test"

date = "13April"
model_path = fr"/storage/model/{date}"

val_accuracy_path = model_path + "/" + r"model_val_accuracy_lite_{epoch:02d}-{val_accuracy:.2f}.keras"
val_loss_path =  model_path + "/" + r"model_val_loss_lite_{epoch:02d}-{val_loss:.2f}.keras"
json_history_path = model_path + "/" + r"history.json"

image_width = 299
image_height = 299
batch_size = 16
epoches = 30

In [ ]:
data_argumentation_layer = tf.keras.Sequential([
      tf.keras.layers.Rescaling(1./255),
      tf.keras.layers.RandomFlip("HORIZONTAL_AND_VERTICAL"),
      tf.keras.layers.RandomRotation(0.9),
      tf.keras.layers.RandomZoom(0.5),
      tf.keras.layers.RandomContrast(0.9)
])

In [ ]:
dataset_train = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_train_file_path_leaf,
    labels="inferred",
    label_mode="categorical",
    seed=123,
    image_size=(image_width, image_height),
    batch_size=batch_size
)

dataset_train = dataset_train.map(lambda x, y: (data_argumentation_layer(x), y))
dataset_train

In [ ]:
dataset_val = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_val_file_path_leaf,
    labels="inferred",
    label_mode="categorical",
    seed=123,
    image_size=(image_width, image_height),
    batch_size=batch_size
)

dataset_val = dataset_val.map(lambda x, y: (data_argumentation_layer(x), y))
dataset_val

In [ ]:
dataset_test = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_test_file_path_leaf,
    labels="inferred",
    label_mode="categorical",
    seed=123,
    image_size=(image_width, image_height),
    batch_size=batch_size
)

dataset_test = dataset_test.map(lambda x, y: (data_argumentation_layer(x), y))
dataset_test

In [ ]:
base_model = tf.keras.applications.inception_v3.InceptionV3(
    include_top=False,
    weights="imagenet",
    input_shape = (299, 299, 3)
)

In [ ]:
for layer in base_model.layers:
  layer.trainable = False
for layer in base_model.layers[-30:]:
  layer.trainable = True

In [ ]:
from tensorflow.keras.models import Model
from keras.layers import Dense, Dropout, GlobalAveragePooling2D
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
output = Dense(80, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.summary()

In [ ]:
from tensorflow.keras.metrics import CategoricalAccuracy
model.compile(optimizer="sgd",
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
import json
import os
from tensorflow.keras.callbacks import Callback, ModelCheckpoint

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint
checkpoint1 = ModelCheckpoint(val_accuracy_path, monitor='accuracy', verbose=2, save_best_only=True, mode='max')

callbacks_list = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=val_loss_path,
        monitor='val_loss', save_best_only=True, verbose=2),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=50,verbose=2),
    checkpoint1,
]

In [ ]:
history = model.fit(dataset_train, validation_data=dataset_val,callbacks=callbacks_list, epochs=epoches)

In [ ]:
# prompt: write code to save all type of required graph, visuals and data for futher research of history variable, histoiry is not saved as json directly use it
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy vs. Epoch')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss vs. Epoch')
plt.savefig(fr"{model_path}/training_history.png") #Save the plot

# Save model architecture
with open(fr"{model_path}/model_architecture.txt", "w") as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))


# Evaluate the model and save results
loss, accuracy = model.evaluate(dataset_test)
evaluation_results = {"loss": loss, "accuracy": accuracy}
with open(fr"{model_path}/evaluation_results.json", "w") as f:
    json.dump(evaluation_results, f)


y_true = []
y_pred = []
for images, labels in dataset_test:
  y_true.extend(np.argmax(labels, axis=1))
  predictions = model.predict(images)
  y_pred.extend(np.argmax(predictions, axis=1))

cm = confusion_matrix(y_true, y_pred)
row_sums = cm.sum(axis=1, keepdims=True)
norm_conf_mx = cm / row_sums
plt.figure(figsize=(10,10))
sns.heatmap(cm, annot=True, fmt='g')
plt.savefig(fr"{model_path}/confusion_matrix.png")
plt.figure(figsize=(10,10))
sns.heatmap(norm_conf_mx, annot=True, fmt='g')
plt.savefig(fr"{model_path}/confusion_matrix_norm.png")